In [1]:
# =====================
# 0. 라이브러리 & 데이터 로드
# =====================
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
import joblib, os
os.makedirs('C:/Users/user/dacon/Smart_Logistics/models', exist_ok=True)
os.makedirs('C:/Users/user/dacon/Smart_Logistics/results', exist_ok=True)

train  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test   = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

# =====================
# 1. Layout merge
# =====================
layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train.merge(layout_clean, on='layout_id', how='left')
test  = test.merge(layout_clean, on='layout_id', how='left')

# =====================
# 2. 인코딩
# =====================
le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET   = 'avg_delay_minutes_next_30m'
ID_COLS  = ['ID', 'layout_id', 'scenario_id']
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]

X = train[feature_cols]
y = train[TARGET]

print(f'train: {train.shape}, test: {test.shape}')
print(f'feature 수: {len(feature_cols)}')

train: (250000, 107), test: (50000, 106)
feature 수: 103


In [2]:
# =====================
# LightGBM v14 예측값
# =====================
lgbm_preds = np.zeros(len(test))
for fold in range(1, 6):
    m = joblib.load(f'C:/Users/user/dacon/Smart_Logistics/models/lgbm_optuna_lr03_fold{fold}.pkl')
    lgbm_preds += m.predict(test[feature_cols]) / 5
    print(f'LightGBM Fold {fold} done')

# =====================
# CatBoost v15 예측값
# =====================
cat_preds = np.zeros(len(test))
for fold in range(1, 6):
    m = joblib.load(f'C:/Users/user/dacon/Smart_Logistics/models/cat_fold{fold}.pkl')
    cat_preds += m.predict(test[feature_cols]) / 5
    print(f'CatBoost Fold {fold} done')

# =====================
# 앙상블 (단순 평균)
# =====================
ensemble_preds = (lgbm_preds + cat_preds) / 2

test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']
submission = pd.DataFrame({
    'ID'                        : test_id,
    'avg_delay_minutes_next_30m': ensemble_preds
})

submission.to_csv('C:/Users/user/dacon/Smart_Logistics/results/submission_v16_lgbm_cat_ensemble.csv', index=False)
print(submission.head())

LightGBM Fold 1 done
LightGBM Fold 2 done
LightGBM Fold 3 done
LightGBM Fold 4 done
LightGBM Fold 5 done
CatBoost Fold 1 done
CatBoost Fold 2 done
CatBoost Fold 3 done
CatBoost Fold 4 done
CatBoost Fold 5 done
            ID  avg_delay_minutes_next_30m
0  TEST_000000                   15.404702
1  TEST_000001                   16.229043
2  TEST_000002                   20.003611
3  TEST_000003                   18.549804
4  TEST_000004                   17.977032
